# Análises de Zika — esquema estrela unificado

Execute após **um** dos pipelines. Ambos publicam o mesmo esquema MySQL.
As consultas e visualizações ficam neste arquivo, separadas da engenharia dos pipelines.
2026 é parcial; contagens descrevem notificações, não incidência nem risco populacional.
Gráficos são exportados para `saidas/analises/`.

In [ ]:
from pathlib import Path
import sys

# Execute na raiz do clone ou em notebooks/ (inclusive no Colab).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pipelines/contrato.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Clone o repositório e abra o notebook dentro dele.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import os
# Crie previamente um banco dedicado ao projeto, usando utf8mb4.
# Configure a conexão antes de executar. A senha vem do ambiente, sem ser salva no notebook.
os.environ.setdefault('MYSQL_HOST', '127.0.0.1')
os.environ.setdefault('MYSQL_PORT', '3306')
os.environ.setdefault('MYSQL_USER', 'root')
os.environ.setdefault('MYSQL_DATABASE', 'zika_dw')
# os.environ['MYSQL_PASSWORD'] = getpass.getpass('Senha MySQL: ')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pipelines.database import connect
output = ROOT / 'saidas/analises'
output.mkdir(parents=True, exist_ok=True)

def query(sql):
    with connect() as connection, connection.cursor() as cursor:
        cursor.execute(sql)
        return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)

tables_out = output / 'tabelas'
tables_out.mkdir(parents=True, exist_ok=True)


## 1. Notificações mensais e sazonalidade

Agrupa por ano e mês reais de DT_NOTIFIC. A janela janeiro–setembro permite comparar os mesmos meses; não garante completude da vigilância. O arquivo de 2026 é parcial. Consulte as tabelas exportadas e a seção 6 do relatório para os números revisados. A concentração mensal não demonstra causa climática nem incidência populacional.

In [ ]:
monthly = query("""SELECT CAST(t.ano AS SIGNED) AS ano, CAST(t.mes AS SIGNED) AS mes, CAST(SUM(f.qt_notificacoes) AS SIGNED) AS notificacoes
FROM fato_notificacao_zika f JOIN dim_tempo t ON t.sk_tempo=f.fk_tempo_notificacao
GROUP BY t.ano,t.mes ORDER BY t.ano,t.mes""")
ax = monthly.pivot(index='mes', columns='ano', values='notificacoes').plot(marker='o', figsize=(12, 4))
ax.set(xlabel='Mês', ylabel='Notificações', title='Série mensal (2026 parcial)')
plt.tight_layout()
plt.savefig(output / '01_sazonalidade.png')
plt.show()
monthly.to_csv(tables_out / '01_sazonalidade.csv', index=False)
display(monthly)

## 2. Gestantes por UF de residência

Soma notificações com CS_GESTANT de 1 a 4 e, entre elas, as classificadas como confirmadas. São registros, não pessoas únicas. NI preserva residência desconhecida. Diferenças entre UFs não demonstram protocolos de triagem, circulação viral ou resultados laboratoriais negativos; classificação descartada não é sinônimo de teste negativo.

In [ ]:
pregnant = query("""SELECT l.sigla_uf, CAST(SUM(f.fl_gestante) AS SIGNED) AS gestantes,
CAST(SUM(f.fl_gestante * f.fl_confirmado) AS SIGNED) AS confirmadas
FROM fato_notificacao_zika f JOIN dim_localidade l ON l.sk_localidade=f.fk_localidade_residencia
GROUP BY l.sigla_uf ORDER BY gestantes DESC""")
ax = pregnant.set_index('sigla_uf')[['gestantes','confirmadas']].plot.bar(figsize=(12,4))
ax.set(xlabel='UF de residência (NI = não informada)', ylabel='Notificações', title='Gestantes notificadas e confirmadas')
plt.tight_layout()
plt.savefig(output / '02_gestantes.png')
plt.show()
pregnant.to_csv(tables_out / '02_gestantes.csv', index=False)
display(pregnant)

## 3. Tempo até notificação e encerramento

Médias calculadas apenas com intervalos de 0 a 365 dias. A tabela inclui denominadores e cobertura para cada métrica. Dados ausentes e encerramentos ainda pendentes podem alterar a comparação regional. As diferenças são descritivas e não demonstram a causa dos prazos.

In [ ]:
delay = query("""SELECT l.regiao_brasil, AVG(f.qt_dias_atraso_notif) AS atraso_notificacao,
AVG(f.qt_dias_encerramento) AS tempo_encerramento,
COUNT(*) AS notificacoes,
COUNT(f.qt_dias_atraso_notif) AS n_atraso_valido,
COUNT(f.qt_dias_encerramento) AS n_encerramento_valido,
100.0 * COUNT(f.qt_dias_atraso_notif) / COUNT(*) AS cobertura_atraso_pct,
100.0 * COUNT(f.qt_dias_encerramento) / COUNT(*) AS cobertura_encerramento_pct
FROM fato_notificacao_zika f JOIN dim_localidade l ON l.sk_localidade=f.fk_localidade_residencia
GROUP BY l.regiao_brasil ORDER BY l.regiao_brasil""")
ax = delay.set_index('regiao_brasil')[['atraso_notificacao', 'tempo_encerramento']].astype(float).plot.barh(figsize=(12, 4))
ax.set(xlabel='Dias (média dos intervalos válidos de 0 a 365)', ylabel='Região', title='Tempos da vigilância')
plt.tight_layout()
plt.savefig(output / '03_tempos.png')
plt.show()
delay.to_csv(tables_out / '03_tempos.csv', index=False)
display(delay)

## 4. Perfil sociodemográfico por faixa etária e sexo

Análise complementar de contagens e confirmações segundo os atributos registrados. A distribuição das notificações não demonstra risco populacional nem busca ativa em gestantes. Categorias ignoradas permanecem na tabela, embora o gráfico destaque feminino e masculino.

In [ ]:
profile = query("""SELECT p.faixa_etaria, p.descricao_sexo,
CAST(SUM(f.qt_notificacoes) AS SIGNED) AS notificacoes,
CAST(SUM(f.fl_confirmado) AS SIGNED) AS confirmados
FROM fato_notificacao_zika f JOIN dim_paciente p ON p.sk_paciente=f.fk_paciente
GROUP BY p.faixa_etaria, p.descricao_sexo ORDER BY p.faixa_etaria, p.descricao_sexo""")

bands = ['0 a 14 anos', '15 a 29 anos', '30 a 44 anos', '45 a 59 anos', '60 anos ou mais']
ax = (profile[profile['descricao_sexo'].isin(['Feminino', 'Masculino'])]
      .pivot(index='faixa_etaria', columns='descricao_sexo', values='notificacoes')
      .reindex(bands).plot.bar())
ax.set(xlabel='Faixa etária', ylabel='Notificações', title='Notificações por faixa etária e sexo')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(output / '04_perfil_demografico.png')
plt.show()
profile.to_csv(tables_out / '04_perfil_faixa_etaria_sexo.csv', index=False)
display(profile)

## 5. Classificação final e critério por ano

Análise complementar dos critérios registrados entre confirmados e da classificação final. O ano usa DT_NOTIFIC; 2026 é parcial. Inconclusivo / Em Investigação é uma categoria do contrato, que também agrega códigos desconhecidos, e não prova que toda ficha esteja aguardando encerramento.

In [ ]:
outcome = query("""SELECT t.ano, d.classificacao_final, d.criterio_confirmacao,
CAST(SUM(f.qt_notificacoes) AS SIGNED) AS notificacoes
FROM fato_notificacao_zika f
JOIN dim_tempo t ON t.sk_tempo=f.fk_tempo_notificacao
JOIN dim_investigacao_desfecho d ON d.sk_desfecho=f.fk_desfecho
GROUP BY t.ano, d.classificacao_final, d.criterio_confirmacao
ORDER BY t.ano, d.classificacao_final, notificacoes DESC""")

ax = (outcome[outcome['classificacao_final'] == 'Confirmado']
      .pivot(index='ano', columns='criterio_confirmacao', values='notificacoes')
      .plot.bar(stacked=True))
ax.set(xlabel='Ano (2026 parcial)', ylabel='Casos confirmados', title='Confirmados por critério e ano')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(output / '05_criterio_desfecho.png')
plt.show()
outcome.to_csv(tables_out / '05_classificacao_criterio_ano.csv', index=False)
display(outcome)

## 6. Diferença entre residência e local de notificação

Análise complementar da proporção de códigos diferentes. Cada indicador usa seu próprio denominador válido e exclui UF desconhecida. Diferença de código não comprova viagem, encaminhamento ou local de infecção.

In [ ]:
mobility = query("""SELECT l.regiao_brasil,
ROUND(AVG(CASE WHEN l.cod_municipio_ibge6 IS NOT NULL AND u.cod_municipio_notificacao IS NOT NULL
THEN l.cod_municipio_ibge6 <> u.cod_municipio_notificacao END) * 100, 2) AS fora_municipio,
ROUND(AVG(CASE WHEN l.sigla_uf <> 'NI' AND u.sigla_uf_notificacao <> 'NI'
THEN l.sigla_uf <> u.sigla_uf_notificacao END) * 100, 2) AS fora_uf,
COUNT(CASE WHEN l.cod_municipio_ibge6 IS NOT NULL AND u.cod_municipio_notificacao IS NOT NULL THEN 1 END) AS n_municipio_valido,
COUNT(CASE WHEN l.sigla_uf <> 'NI' AND u.sigla_uf_notificacao <> 'NI' THEN 1 END) AS n_uf_valida
FROM fato_notificacao_zika f
JOIN dim_localidade l ON l.sk_localidade=f.fk_localidade_residencia
JOIN dim_unidade_notificadora u ON u.sk_unidade=f.fk_unidade_notificadora
GROUP BY l.regiao_brasil ORDER BY fora_municipio DESC""")

ax = mobility.set_index('regiao_brasil')[['fora_municipio', 'fora_uf']].astype(float).plot.barh(figsize=(12, 4))
ax.set(xlabel='Percentual de notificações (%)', ylabel='Região', title='Notificação em município ou UF diferente da residência')
plt.tight_layout()
plt.savefig(output / '06_mobilidade_assistencial.png')
plt.show()
mobility.to_csv(tables_out / '06_mobilidade_assistencial_regiao.csv', index=False)
display(mobility)